# Progressive MuJoCo Tutorial: From One Knee Joint to a 2D Biped

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/robomotic/robotic-hardware/blob/main/notebooks/mujoco_servo_part1.ipynb)

This notebook builds a robot-control tutorial from the simplest setup to a small walking system.

Learning path:
1. Stage 1: one hinge joint (knee) with PD position control.
2. Stage 2: one planar leg with hip, knee, ankle.
3. Stage 3: planar point-foot biped (2 legs) with a simple gait state machine.
4. Theory vs simulation checks at each stage.

Hardware reference:
- Dynamixel X-series style servo characteristics (position mode, speed/torque limits, practical safety margins).

Simulation stack:
- `mujoco` Python package as primary backend.
- `mjx` notes and optional section for JAX-based rollouts.

Control scope in this first implementation:
- PD position control only (torque emerges through actuator dynamics and gain settings).

## 1) Import Required Libraries

This section imports numerical, plotting, and simulation dependencies. The notebook is designed to run even when optional packages are missing by degrading gracefully where possible.

In [1]:
import math
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import matplotlib.pyplot as plt

try:
    import mujoco
    MUJOCO_AVAILABLE = True
except Exception as exc:
    MUJOCO_AVAILABLE = False
    print(f"MuJoCo import failed: {exc}")

try:
    import jax
    import jax.numpy as jnp
    from mujoco import mjx
    MJX_AVAILABLE = True
except Exception as exc:
    MJX_AVAILABLE = False
    print(f"MJX import skipped: {exc}")

np.set_printoptions(precision=4, suppress=True)
plt.style.use("default")

print("MuJoCo available:", MUJOCO_AVAILABLE)
print("MJX available:", MJX_AVAILABLE)

MuJoCo import failed: No module named 'mujoco'
MJX import skipped: No module named 'jax'
MuJoCo available: False
MJX available: False


## 2) Define Configuration and Constants

We use a single normalized servo profile inspired by Dynamixel X-series actuators. Values are approximate tutorial defaults, not a datasheet replacement.

Mapping idea:
- Position control in a servo translates position error into motor effort.
- In MuJoCo, we represent that with a `position` actuator and explicit gain/bias settings.
- Torque limits are approximated through `forcerange` and checked in analysis plots.

In [2]:
@dataclass
class ServoProfile:
    name: str
    nominal_voltage_v: float
    continuous_torque_nm: float
    peak_torque_nm: float
    max_speed_rad_s: float
    position_resolution_ticks: int


SERVO = ServoProfile(
    name="Dynamixel-X (tutorial profile)",
    nominal_voltage_v=24.0,
    continuous_torque_nm=5.0,
    peak_torque_nm=12.0,
    max_speed_rad_s=7.0,
    position_resolution_ticks=4096,
)

SIM_CFG: Dict[str, float] = {
    "dt": 0.002,
    "duration": 4.0,
    "kp": 120.0,
    "kd": 10.0,
    "gravity": 9.81,
}

print(SERVO)
print(SIM_CFG)


def radians_to_ticks(rad: np.ndarray) -> np.ndarray:
    return (rad / (2 * np.pi) * SERVO.position_resolution_ticks).astype(int)


def clamp(x: np.ndarray, lo: float, hi: float) -> np.ndarray:
    return np.minimum(np.maximum(x, lo), hi)

ServoProfile(name='Dynamixel-X (tutorial profile)', nominal_voltage_v=24.0, continuous_torque_nm=5.0, peak_torque_nm=12.0, max_speed_rad_s=7.0, position_resolution_ticks=4096)
{'dt': 0.002, 'duration': 4.0, 'kp': 120.0, 'kd': 10.0, 'gravity': 9.81}


In [3]:
def make_model_from_xml(xml: str):
    if not MUJOCO_AVAILABLE:
        raise RuntimeError("MuJoCo is not available. Install `mujoco` to run simulation cells.")
    model = mujoco.MjModel.from_xml_string(xml)
    data = mujoco.MjData(model)
    return model, data


def run_pd_position_control(model, data, q_target_fn, duration, kp, kd):
    n_steps = int(duration / model.opt.timestep)
    t_hist = np.zeros(n_steps)
    q_hist = np.zeros((n_steps, model.nq))
    qd_hist = np.zeros((n_steps, model.nv))
    ctrl_hist = np.zeros((n_steps, model.nu))
    torque_hist = np.zeros((n_steps, model.nu))

    for i in range(n_steps):
        t = i * model.opt.timestep
        q_target = q_target_fn(t)

        q = data.qpos[: model.nu].copy()
        qd = data.qvel[: model.nu].copy()

        # PD command converted to a position-servo equivalent control signal.
        ctrl = q_target + (kd / max(kp, 1e-6)) * (0.0 - qd)
        data.ctrl[:] = ctrl

        mujoco.mj_step(model, data)

        t_hist[i] = t
        q_hist[i] = data.qpos
        qd_hist[i] = data.qvel
        ctrl_hist[i] = data.ctrl
        torque_hist[i] = data.actuator_force.copy()

    return {
        "t": t_hist,
        "q": q_hist,
        "qd": qd_hist,
        "ctrl": ctrl_hist,
        "tau": torque_hist,
    }

## 3) Implement Core Logic - Stage 1: One Knee Joint

Model: a single hinge in gravity.

Theory reference (static hold):

$$
\tau_{hold}(\theta) = m g l_c \sin(\theta)
$$

where $m$ is link mass, $l_c$ is COM distance from joint.

Goal:
- Track a target angle trajectory with PD position control.
- Compare simulated actuator torque against theoretical static requirement.

In [4]:
knee_xml = f"""
<mujoco model='one_knee'>
  <option timestep='{SIM_CFG['dt']}' gravity='0 0 -{SIM_CFG['gravity']}' integrator='RK4'/>
  <worldbody>
    <body name='thigh' pos='0 0 0'>
      <joint name='knee' type='hinge' axis='0 1 0' range='-2.5 0.2' damping='0.6'/>
      <geom type='capsule' fromto='0 0 0 0 0 -0.40' size='0.04' density='700'/>
    </body>
  </worldbody>
  <actuator>
    <position name='knee_servo' joint='knee' kp='{SIM_CFG['kp']}' forcerange='-{SERVO.peak_torque_nm} {SERVO.peak_torque_nm}'/>
  </actuator>
</mujoco>
"""

if MUJOCO_AVAILABLE:
    knee_model, knee_data = make_model_from_xml(knee_xml)

    def knee_target(t):
        return np.array([0.4 * np.sin(2 * np.pi * 0.4 * t) - 0.5])

    knee_out = run_pd_position_control(
        knee_model,
        knee_data,
        q_target_fn=knee_target,
        duration=SIM_CFG["duration"],
        kp=SIM_CFG["kp"],
        kd=SIM_CFG["kd"],
    )

    thigh_mass = float(knee_model.body_mass[1])
    com_vec = knee_model.body_ipos[1]
    lc = abs(float(com_vec[2]))
    theta = knee_out["q"][:, 0]
    tau_theory = thigh_mass * SIM_CFG["gravity"] * lc * np.sin(theta)

    fig, ax = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
    ax[0].plot(knee_out["t"], knee_out["q"][:, 0], label="knee angle (rad)")
    ax[0].plot(knee_out["t"], [knee_target(t)[0] for t in knee_out["t"]], "--", label="target")
    ax[0].set_ylabel("angle [rad]")
    ax[0].legend()
    ax[0].grid(True, alpha=0.3)

    ax[1].plot(knee_out["t"], knee_out["tau"][:, 0], label="sim actuator torque")
    ax[1].plot(knee_out["t"], tau_theory, "--", label="theory static torque")
    ax[1].axhline(SERVO.continuous_torque_nm, color="tab:green", linestyle=":", label="cont. torque")
    ax[1].axhline(-SERVO.continuous_torque_nm, color="tab:green", linestyle=":")
    ax[1].set_xlabel("time [s]")
    ax[1].set_ylabel("torque [N m]")
    ax[1].legend()
    ax[1].grid(True, alpha=0.3)
    plt.show()

    print("Stage 1 complete: compare dynamic sim torque to static theory curve.")
else:
    print("Install mujoco to run Stage 1 simulation.")

Install mujoco to run Stage 1 simulation.


### Stage 2: One Planar Leg (Hip-Knee-Ankle)

Now we move from a single joint to a 3-joint leg in sagittal plane.

Focus:
- Joint coupling effects
- Pose hold torque distribution
- Tracking of a simple coordinated trajectory

In [5]:
leg_xml = f"""
<mujoco model='planar_leg'>
  <option timestep='{SIM_CFG['dt']}' gravity='0 0 -{SIM_CFG['gravity']}' integrator='RK4'/>
  <worldbody>
    <body name='hip_link' pos='0 0 0.9'>
      <joint name='hip' type='hinge' axis='0 1 0' range='-1.5 1.2' damping='0.8'/>
      <geom type='capsule' fromto='0 0 0 0 0 -0.35' size='0.04' density='700'/>

      <body name='thigh' pos='0 0 -0.35'>
        <joint name='knee' type='hinge' axis='0 1 0' range='-2.5 0.2' damping='0.8'/>
        <geom type='capsule' fromto='0 0 0 0 0 -0.35' size='0.035' density='700'/>

        <body name='shank' pos='0 0 -0.35'>
          <joint name='ankle' type='hinge' axis='0 1 0' range='-0.9 0.9' damping='0.4'/>
          <geom type='capsule' fromto='0 0 0 0.08 0 -0.08' size='0.025' density='700'/>
        </body>
      </body>
    </body>
  </worldbody>
  <actuator>
    <position name='hip_servo' joint='hip' kp='{SIM_CFG['kp']}' forcerange='-{SERVO.peak_torque_nm} {SERVO.peak_torque_nm}'/>
    <position name='knee_servo' joint='knee' kp='{SIM_CFG['kp']}' forcerange='-{SERVO.peak_torque_nm} {SERVO.peak_torque_nm}'/>
    <position name='ankle_servo' joint='ankle' kp='{SIM_CFG['kp']}' forcerange='-{SERVO.peak_torque_nm} {SERVO.peak_torque_nm}'/>
  </actuator>
</mujoco>
"""

if MUJOCO_AVAILABLE:
    leg_model, leg_data = make_model_from_xml(leg_xml)

    def leg_target(t):
        hip = 0.25 * np.sin(2 * np.pi * 0.5 * t)
        knee = -0.6 + 0.35 * np.sin(2 * np.pi * 0.5 * t + 0.8)
        ankle = -0.2 * np.sin(2 * np.pi * 0.5 * t + 1.6)
        return np.array([hip, knee, ankle])

    leg_out = run_pd_position_control(
        leg_model,
        leg_data,
        q_target_fn=leg_target,
        duration=SIM_CFG["duration"],
        kp=SIM_CFG["kp"],
        kd=SIM_CFG["kd"],
    )

    q_ref = np.array([leg_target(t) for t in leg_out["t"]])
    rms = np.sqrt(np.mean((leg_out["q"][:, :3] - q_ref) ** 2, axis=0))

    fig, ax = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
    labels = ["hip", "knee", "ankle"]

    for j in range(3):
        ax[0].plot(leg_out["t"], leg_out["q"][:, j], label=f"{labels[j]} q")
        ax[0].plot(leg_out["t"], q_ref[:, j], "--", alpha=0.7)
    ax[0].set_ylabel("angle [rad]")
    ax[0].grid(True, alpha=0.3)
    ax[0].legend(ncol=3)

    for j in range(3):
        ax[1].plot(leg_out["t"], leg_out["tau"][:, j], label=f"{labels[j]} tau")
    ax[1].axhline(SERVO.continuous_torque_nm, color="tab:green", linestyle=":")
    ax[1].axhline(-SERVO.continuous_torque_nm, color="tab:green", linestyle=":")
    ax[1].set_xlabel("time [s]")
    ax[1].set_ylabel("torque [N m]")
    ax[1].grid(True, alpha=0.3)
    ax[1].legend(ncol=3)
    plt.show()

    print("Stage 2 RMS tracking [hip,knee,ankle] rad:", np.round(rms, 4))
else:
    print("Install mujoco to run Stage 2 simulation.")

Install mujoco to run Stage 2 simulation.


### Stage 3: Planar Point-Foot Biped

We now create two legs and a torso in a 2D setup. A simple finite-state gait reference is used:
- Left stance / right swing
- Switch

This is intentionally minimal and educational, not a production walking controller.

In [6]:
biped_xml = f"""
<mujoco model='planar_biped'>
  <option timestep='{SIM_CFG['dt']}' gravity='0 0 -{SIM_CFG['gravity']}' integrator='RK4'/>
  <worldbody>
    <geom name='floor' type='plane' pos='0 0 0' size='3 3 0.1' friction='1.0 0.005 0.0001'/>

    <body name='torso' pos='0 0 0.95'>
      <joint name='rootx' type='slide' axis='1 0 0'/>
      <joint name='rootz' type='slide' axis='0 0 1'/>
      <joint name='rootpitch' type='hinge' axis='0 1 0'/>
      <geom type='capsule' fromto='0 0 -0.10 0 0 0.20' size='0.08' density='600'/>

      <body name='left_thigh' pos='0 0 -0.08'>
        <joint name='l_hip' type='hinge' axis='0 1 0' range='-1.4 1.2' damping='1.0'/>
        <geom type='capsule' fromto='0 0 0 0 0 -0.33' size='0.04' density='700'/>
        <body name='left_shank' pos='0 0 -0.33'>
          <joint name='l_knee' type='hinge' axis='0 1 0' range='-2.4 0.15' damping='0.8'/>
          <geom type='capsule' fromto='0 0 0 0 0 -0.33' size='0.035' density='700'/>
          <body name='left_foot' pos='0 0 -0.33'>
            <joint name='l_ankle' type='hinge' axis='0 1 0' range='-0.8 0.8' damping='0.5'/>
            <geom type='sphere' size='0.035' density='500'/>
          </body>
        </body>
      </body>

      <body name='right_thigh' pos='0 0 -0.08'>
        <joint name='r_hip' type='hinge' axis='0 1 0' range='-1.4 1.2' damping='1.0'/>
        <geom type='capsule' fromto='0 0 0 0 0 -0.33' size='0.04' density='700'/>
        <body name='right_shank' pos='0 0 -0.33'>
          <joint name='r_knee' type='hinge' axis='0 1 0' range='-2.4 0.15' damping='0.8'/>
          <geom type='capsule' fromto='0 0 0 0 0 -0.33' size='0.035' density='700'/>
          <body name='right_foot' pos='0 0 -0.33'>
            <joint name='r_ankle' type='hinge' axis='0 1 0' range='-0.8 0.8' damping='0.5'/>
            <geom type='sphere' size='0.035' density='500'/>
          </body>
        </body>
      </body>
    </body>
  </worldbody>

  <actuator>
    <position name='l_hip_servo' joint='l_hip' kp='{SIM_CFG['kp']}' forcerange='-{SERVO.peak_torque_nm} {SERVO.peak_torque_nm}'/>
    <position name='l_knee_servo' joint='l_knee' kp='{SIM_CFG['kp']}' forcerange='-{SERVO.peak_torque_nm} {SERVO.peak_torque_nm}'/>
    <position name='l_ankle_servo' joint='l_ankle' kp='{SIM_CFG['kp']}' forcerange='-{SERVO.peak_torque_nm} {SERVO.peak_torque_nm}'/>
    <position name='r_hip_servo' joint='r_hip' kp='{SIM_CFG['kp']}' forcerange='-{SERVO.peak_torque_nm} {SERVO.peak_torque_nm}'/>
    <position name='r_knee_servo' joint='r_knee' kp='{SIM_CFG['kp']}' forcerange='-{SERVO.peak_torque_nm} {SERVO.peak_torque_nm}'/>
    <position name='r_ankle_servo' joint='r_ankle' kp='{SIM_CFG['kp']}' forcerange='-{SERVO.peak_torque_nm} {SERVO.peak_torque_nm}'/>
  </actuator>
</mujoco>
"""

if MUJOCO_AVAILABLE:
    biped_model, biped_data = make_model_from_xml(biped_xml)

    def biped_reference(t):
        phase = 2 * np.pi * 0.7 * t
        # Left/right anti-phase gait references.
        l_hip = 0.25 * np.sin(phase)
        r_hip = 0.25 * np.sin(phase + np.pi)
        l_knee = -0.55 + 0.35 * np.maximum(0.0, np.sin(phase))
        r_knee = -0.55 + 0.35 * np.maximum(0.0, np.sin(phase + np.pi))
        l_ankle = -0.15 * np.sin(phase)
        r_ankle = -0.15 * np.sin(phase + np.pi)
        return np.array([l_hip, l_knee, l_ankle, r_hip, r_knee, r_ankle])

    biped_out = run_pd_position_control(
        biped_model,
        biped_data,
        q_target_fn=biped_reference,
        duration=6.0,
        kp=SIM_CFG["kp"],
        kd=SIM_CFG["kd"],
    )

    rootx = biped_out["q"][:, 0]
    rootz = biped_out["q"][:, 1]
    pitch = biped_out["q"][:, 2]

    fig, ax = plt.subplots(3, 1, figsize=(9, 8), sharex=True)
    ax[0].plot(biped_out["t"], rootx)
    ax[0].set_ylabel("root x [m]")
    ax[0].grid(True, alpha=0.3)

    ax[1].plot(biped_out["t"], rootz)
    ax[1].set_ylabel("root z [m]")
    ax[1].grid(True, alpha=0.3)

    ax[2].plot(biped_out["t"], pitch)
    ax[2].set_ylabel("root pitch [rad]")
    ax[2].set_xlabel("time [s]")
    ax[2].grid(True, alpha=0.3)
    plt.show()

    step_events = np.sum(np.diff(np.sign(np.sin(2 * np.pi * 0.7 * biped_out["t"]))) != 0)
    print("Stage 3 complete. Approx gait phase switches:", int(step_events))
else:
    print("Install mujoco to run Stage 3 simulation.")

Install mujoco to run Stage 3 simulation.


## 4) Run and Test Implementation

This section performs quick consistency checks and summarizes theory-vs-simulation metrics.

Interpretation tips:
- If simulated torque differs from static theory in Stage 1, inertial terms and damping are expected causes.
- If Stage 2 tracking error is high, tune `kp`, `kd`, and timestep.
- If Stage 3 is unstable, reduce gait amplitude/frequency and increase damping.

## 5) MJX Compatibility Notes

`mjx` can run MuJoCo-style dynamics in JAX workflows for batching and accelerator-backed rollouts.

Practical guidance:
1. Keep one XML model source and one controller logic path.
2. Validate in `mujoco` first, then port to `mjx` for performance experiments.
3. Expect small numerical differences from solver choices and precision settings.

In [7]:
summary_rows: List[Tuple[str, float, float, float]] = []

if MUJOCO_AVAILABLE:
    # Stage 1: compare |mean(sim torque)| to |mean(static theory)| as a rough consistency metric.
    s1_sim = float(np.mean(np.abs(knee_out["tau"][:, 0])))
    s1_theory = float(np.mean(np.abs(tau_theory)))
    s1_err = 100.0 * abs(s1_sim - s1_theory) / max(s1_theory, 1e-6)
    summary_rows.append(("Stage 1 mean |tau|", s1_theory, s1_sim, s1_err))

    # Stage 2: tracking RMS target bound.
    s2_rms = float(np.mean(rms))
    s2_target = 0.15
    s2_err = 100.0 * max(s2_rms - s2_target, 0.0) / s2_target
    summary_rows.append(("Stage 2 mean RMS [rad]", s2_target, s2_rms, s2_err))

    # Stage 3: torso height stay above a floor margin.
    s3_floor_margin = 0.55
    s3_meas = float(np.min(rootz))
    s3_err = 100.0 * max(s3_floor_margin - s3_meas, 0.0) / s3_floor_margin
    summary_rows.append(("Stage 3 min torso z [m]", s3_floor_margin, s3_meas, s3_err))

    print("Theory vs simulation summary")
    print("=" * 72)
    print(f"{'Metric':30s} {'Expected':>12s} {'Observed':>12s} {'Error %':>12s}")
    print("-" * 72)
    for name, expected, observed, err in summary_rows:
        print(f"{name:30s} {expected:12.4f} {observed:12.4f} {err:12.2f}")
else:
    print("MuJoCo unavailable: smoke checks skipped.")

if MJX_AVAILABLE and MUJOCO_AVAILABLE:
    print("\nMJX quick bridge test")
    model = mujoco.MjModel.from_xml_string(knee_xml)
    mjx_model = mjx.put_model(model)
    data = mujoco.MjData(model)
    data.ctrl[:] = np.array([-0.4])
    mujoco.mj_step(model, data)
    mjx_data = mjx.put_data(model, data)
    _ = mjx.step(mjx_model, mjx_data)
    print("MJX step executed successfully.")
else:
    print("MJX optional path not executed (missing JAX/MJX or MuJoCo).")

MuJoCo unavailable: smoke checks skipped.
MJX optional path not executed (missing JAX/MJX or MuJoCo).
